# A first step in web scraping

Each cell does one thing. Run them from top to bottom.

First a small HTML page we write ourselves, then one real book site. At the bottom, the same job becomes a function you call with parameters.

In [1]:
import csv
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

## 1. HTML is text

The page you see in a browser is text made of tags. Read this sample as boxes:

- `<h1>` is the page title
- each `<div class="book">` is one book
- `<h2>` is the title, `<p class="author">` is the author, `<p class="price">` is the price
- Hamlet has no price tag, on purpose

In [2]:
html = """
<html>
  <body>
    <h1>Campus Bookstore</h1>

    <div class="book">
      <h2>Dune</h2>
      <p class="author">Frank Herbert</p>
      <p class="price">12.50</p>
    </div>

    <div class="book">
      <h2>1984</h2>
      <p class="author">George Orwell</p>
      <p class="price">9.00</p>
    </div>

    <div class="book">
      <h2>Hamlet</h2>
      <p class="author">William Shakespeare</p>
    </div>
  </body>
</html>
"""
print(html)


<html>
  <body>
    <h1>Campus Bookstore</h1>

    <div class="book">
      <h2>Dune</h2>
      <p class="author">Frank Herbert</p>
      <p class="price">12.50</p>
    </div>

    <div class="book">
      <h2>1984</h2>
      <p class="author">George Orwell</p>
      <p class="price">9.00</p>
    </div>

    <div class="book">
      <h2>Hamlet</h2>
      <p class="author">William Shakespeare</p>
    </div>
  </body>
</html>



## 2. Find the title

Beautiful Soup turns that text into a tree. `find` returns the first matching tag.

In [3]:
soup = BeautifulSoup(html, "html.parser")
print(soup.find("h1").get_text(strip=True))

Campus Bookstore


## 3. Look inside one card

Write `class_`, because `class` is already a Python keyword.

In [4]:
card = soup.find("div", class_="book")
print(card.find("h2").get_text(strip=True))
print(card.find("p", class_="author").get_text(strip=True))
print(card.find("p", class_="price").get_text(strip=True))

Dune
Frank Herbert
12.50


## 4. All the cards

`find_all` returns every match as a list. Hamlet has no price tag. Calling `.get_text()` on a missing tag causes an error, so check first.

In [5]:
for card in soup.find_all("div", class_="book"):
    title = card.find("h2").get_text(strip=True)
    author = card.find("p", class_="author").get_text(strip=True)
    price = card.find("p", class_="price")
    if price is None:
        price = ""
    else:
        price = price.get_text(strip=True)
    print(title, "|", author, "|", price)

Dune | Frank Herbert | 12.50
1984 | George Orwell | 9.00
Hamlet | William Shakespeare | 


## 5. Fetch a real page

`requests` goes to the address and brings back the HTML. This site is published for scraping practice. Before you point the same code at another site, check that the site allows it.

In [6]:
url = "https://books.toscrape.com/"
response = requests.get(
    url,
    headers={"User-Agent": "DataScienceClassBot/1.0 (education)"},
    timeout=15,
)
response.raise_for_status()
response.encoding = "utf-8"
page = BeautifulSoup(response.text, "html.parser")
print(response.status_code)
print(page.find("title").get_text(strip=True))

200
All products | Books to Scrape - Sandbox


## 6. One book on the site

The card selector is `article.product_pod`. The title looks cut off on the page. The full title is in the link's `title` attribute.

In [7]:
card = page.select_one("article.product_pod")
link = card.select_one("h3 a")
print("visible:", link.get_text(strip=True))
print("full title:", link.get("title"))
print("price:", card.select_one("p.price_color").get_text(strip=True))

visible: A Light in the ...
full title: A Light in the Attic
price: £51.77


## 7. Every book on the page

In [8]:
books = []
for card in page.select("article.product_pod"):
    link = card.select_one("h3 a")
    books.append({
        "title": link.get("title"),
        "price": card.select_one("p.price_color").get_text(strip=True),
    })

print("count:", len(books))
print(books[0])
print(books[1])

count: 20
{'title': 'A Light in the Attic', 'price': '£51.77'}
{'title': 'Tipping the Velvet', 'price': '£53.74'}


## 8. The next page

The address of the next page is a link inside the page. `urljoin` turns a relative link into a full address.

In [9]:
next_link = page.select_one("li.next a")
next_url = urljoin(response.url, next_link.get("href"))
print(next_url)

https://books.toscrape.com/catalogue/page-2.html


## 9. Save every book

Follow the next-page link until it disappears, then write every row to a CSV file.

`("h3 a", None)` reads the visible text. `("h3 a", "title")` reads that attribute. For another site, change only the values in the call below.

In [10]:
def scrape(url, card, fields, next_link=None, delay=1, csv_path="output.csv"):
    records = []
    address = url
    seen = set()

    while address not in seen:
        seen.add(address)
        response = requests.get(
            address,
            headers={"User-Agent": "DataScienceClassBot/1.0 (education)"},
            timeout=15,
        )
        response.raise_for_status()
        response.encoding = "utf-8"
        page = BeautifulSoup(response.text, "html.parser")
        print("page", len(seen))

        for block in page.select(card):
            row = {}
            for column, (selector, attribute) in fields.items():
                tag = block.select_one(selector)
                if tag is None:
                    row[column] = ""
                elif attribute:
                    row[column] = tag.get(attribute) or ""
                else:
                    row[column] = tag.get_text(strip=True)
            records.append(row)

        if not next_link:
            break
        link = page.select_one(next_link)
        if link is None or not link.get("href"):
            break
        address = urljoin(response.url, link.get("href"))
        time.sleep(delay)

    with open(csv_path, "w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=list(fields.keys()))
        writer.writeheader()
        writer.writerows(records)

    print(len(records), "rows written to", csv_path)
    return records

In [11]:
records = scrape(
    url="https://books.toscrape.com/",
    card="article.product_pod",
    fields={
        "title": ("h3 a", "title"),
        "price": ("p.price_color", None),
    },
    next_link="li.next a",
    delay=1,
    csv_path="books.csv",
)
records[:3]

page 1
page 2
page 3
page 4
page 5
page 6
page 7
page 8
page 9
page 10
page 11
page 12
page 13
page 14
page 15
page 16
page 17
page 18
page 19
page 20
page 21
page 22
page 23
page 24
page 25
page 26
page 27
page 28
page 29
page 30
page 31
page 32
page 33
page 34
page 35
page 36
page 37
page 38
page 39
page 40
page 41
page 42
page 43
page 44
page 45
page 46
page 47
page 48
page 49
page 50
1000 rows written to books.csv


[{'title': 'A Light in the Attic', 'price': '£51.77'},
 {'title': 'Tipping the Velvet', 'price': '£53.74'},
 {'title': 'Soumission', 'price': '£50.10'}]

In [12]:
import pandas as pd

csv = pd.read_csv("books.csv")
csv.head(20)


,title,price
0,A Light in the Attic,£51.77
1,Tipping the Velvet,£53.74
2,Soumission,£50.10
3,Sharp Objects,£47.82
4,Sapiens: A Brief History of Humankind,£54.23
5,The Requiem Red,£22.65
6,The Dirty Little Secrets of Getting Your Dream...,£33.34
7,The Coming Woman: A Novel Based on the Life of...,£17.93
8,The Boys in the Boat: Nine Americans and Their...,£22.60
9,The Black Maria,£52.15


In [13]:
csv.describe()

,title,price
count,1000,1000
unique,999,903
top,The Star-Touched Queen,£44.18
freq,2,3
